# 🧬 Notebook 1 — Explore DNA Data and Fine-Tune DNABERT

## The question for today

> **Can a pretrained DNA language model learn to distinguish CTCF-binding DNA from background DNA?**

This notebook is designed for students coming from **different backgrounds**. You may have experience in computer science, biology, both, or neither.

You are **not expected to memorize every line of Python**. Instead, you will repeatedly practice the same workflow:

**ask a question → inspect data → make a graph → run a model → change something → compare results**

### By the end of Notebook 1, you should be able to

- Explain the CTCF binding prediction problem.
- Load a biological dataset into a `pandas` DataFrame.
- Use `head()`, `value_counts()`, `describe()`, and `groupby()` to explore data.
- Make simple bar, histogram, box, line, and scatter plots.
- Explain **6-mer tokenization → token IDs → embeddings → Transformer → [CLS] → classifier**.
- Explain what happens during one neural-network training step.
- Fine-tune DNABERT.
- Deliberately modify important fine-tuning settings.
- Save and compare multiple experimental runs.

### How to read this notebook

| Marker | Meaning |
|---|---|
| 🔒 **RUN ONLY** | Infrastructure. Run it; you do not need to memorize it. |
| 👀 **READ** | Important code. Read the comments and follow the main idea. |
| 🧠 **BUILD IT** | A core concept turned into code. Read this one closely. |
| ✏️ **EDIT ME** | Change a value, re-run, and see what moves. |
| 🔲 **YOUR TURN** | A line is left blank on purpose. Write it, then run the check cell below it. |
| ✅ **CHECKPOINT** | Stop and answer before moving on. |

Every notebook in this bootcamp uses these same six markers.


> **New to Python or machine learning?** Work through
> **`Notebook_Start_Here.ipynb`** first — about an hour, and it teaches
> exactly the Python this notebook uses, plus a glossary you can keep open
> in another tab.

## Before you start

**What this notebook is for:** the main event. You explore real data, make graphs, and fine-tune your first language model on DNA.

**What you will leave with:**

1. **You will have trained a model** and watched its numbers improve
   epoch by epoch.
2. You can read a training log and tell whether it went well or badly —
   including a real example of a run that destroyed itself.
3. You know why a single accuracy number is not enough, and what to ask
   instead.
4. You have compared your model against a deliberately stupid baseline —
   the habit that stops you fooling yourself.

**New words you will meet here:** fine-tuning, epoch, batch, learning rate, AUROC, confusion matrix, validation set

**If you get lost:** the ✅ CHECKPOINT cells are the spine of the notebook. If you can answer those, you are following it, whatever the code is doing.

**Time:** about 3 hours. Sections 1-4 are exploration, 5-8 are the model, 9 onward is judging the results. Section 9 is where the real thinking happens.

## Our roadmap

```mermaid
flowchart LR
    A["Biological question"] --> B["Explore dataset"]
    B --> C["Clean + split data"]
    C --> D["DNA → 6-mers"]
    D --> E["DNABERT"]
    E --> F["Fine-tune"]
    F --> G["Make graphs"]
    G --> H["Change parameters"]
    H --> I["Compare experiments"]
```

The goal is not to find a single “magic” configuration. The goal is to learn how to **ask controlled questions with data and models**.

## Where this notebook is going

Four movements. If you get lost, come back here.

| Part | Sections | What you are doing | What you end up with |
|---|---|---|---|
| **1. The question** | 1 | Biology: what is CTCF, where do labels come from | A binary classification problem |
| **2. Look at the data** | 2-4 | pandas, five kinds of graph, audit and clean | A dataset you trust, and its GC-content quirk |
| **3. The model** | 5-8 | 6-mers → tokens → DNABERT → classifier head, then fine-tune | A trained model and a saved run record |
| **4. What did you get?** | 9-13 | Eight metrics, four diagnostic plots, controlled experiments | An answer to "is it actually good, and why" |

**Notebook 2** opens the model up and rebuilds it from scratch.
**Notebook 3** takes the same job to many GPUs.

You are not expected to memorize Python syntax. You *are* expected to be able
to run this loop yourself by the end:

> ask a question → inspect the data → make a graph → run a model →
> change one thing → compare

## 🧭 Python survival guide — read this once, then come back when needed

You do **not** need to memorize Python syntax. When you see unfamiliar code, first identify the job it is doing.

| Python word | Plain-English meaning | Tiny example |
|---|---|---|
| **variable** | A name that stores a value | `k = 6` |
| **function** | A reusable mini-program that performs one job | `gc_content(sequence)` |
| **argument** | A value you give to a function | `gc_content("ACGT")` |
| **return** | The value a function gives back | `return gc_fraction` |
| **list** | An ordered collection | `["A", "C", "G", "T"]` |
| **dictionary (`dict`)** | Named values stored as key → value pairs | `{"A": 1, "C": 2}` |
| **DataFrame** | A pandas table: rows are examples, columns are properties | `df.head()` |
| **boolean mask** | A True/False filter that selects rows | `df[df["label"] == 1]` |
| **class** | A blueprint for an object that stores data and behavior together | `class DNASet(...)` |
| **method** | A function that belongs to an object/class | `model.forward(...)` |

### How to read a function

```python
def gc_content(sequence):      # function name + input
    gc = ...                   # work done inside the function
    return gc                  # value sent back
```

Read that as:

> “Given a `sequence`, calculate something called `gc`, then give `gc` back.”

### How to read a class

```python
class ExampleModel(nn.Module):
    def __init__(self):
        ...

    def forward(self, x):
        ...
```

- `__init__` = **what pieces does this object contain?**
- `forward` = **what happens to the input when it moves through the model?**
- `self` = **this particular object**. You normally do not pass it yourself.

Whenever a cell is marked **🔒 RUN ONLY**, focus on the explanation above it rather than every Python detail.


# 1. The Biology — Just Enough to Ask the Question

DNA is an ordered sequence made from four letters:

- **A** — adenine
- **C** — cytosine
- **G** — guanine
- **T** — thymine

A **transcription factor** is a protein that can bind particular regions of DNA. One important DNA-binding protein is **CTCF**.

For this notebook, we simplify the biological question into a binary classification problem:

| Label | Meaning |
|---|---|
| `1` | CTCF-binding DNA |
| `0` | Background DNA |

So the machine-learning problem is:

> Given a DNA sequence, predict **Binding** or **Background**.

```mermaid
flowchart LR
    A["DNA sequence"] --> B["Model"]
    B --> C{"Prediction"}
    C -->|"1"| D["CTCF Binding"]
    C -->|"0"| E["Background"]
```

The model does **not** directly observe a CTCF protein. It learns statistical sequence patterns that help distinguish the two classes.

## Where do binding labels come from?

Experiments such as **ChIP-seq** can identify genome regions associated with a particular DNA-binding protein.

A simplified view is:

```mermaid
flowchart TD
    A["Living cells"] --> B["Crosslink proteins to DNA"]
    B --> C["Fragment the DNA"]
    C --> D["Use a CTCF antibody<br/>to capture CTCF + attached DNA"]
    D --> E["Sequence the captured DNA"]
    E --> F["Identify genomic regions<br/>associated with CTCF"]
```

Those experimentally identified regions can provide positive examples for a classifier.

### ✅ CHECKPOINT 1

Explain the task to a partner without using the words *Transformer* or *DNABERT*:

1. What is the input?
2. What are the two possible labels?
3. What biological phenomenon are we trying to predict?

# 2. Meet the Dataset with Python

Before using a neural network, we should understand the data ourselves.

We will use **pandas**, a Python library for working with tables.

A pandas table is called a **DataFrame**.

Think of it like a spreadsheet:

- each **row** = one example
- each **column** = one property of that example

In [ ]:
# 🔒 RUN ONLY — imports, paths, and reproducibility
import os
import time
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
    confusion_matrix, precision_score, recall_score, f1_score, roc_curve,
    precision_recall_curve)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# All bootcamp notebooks resolve the SAME folder, so a dataset built in
# Notebook 0 is found by Notebooks 1-3 even if you launch them from
# elsewhere. Override by setting the DNA_BOOTCAMP_HOME environment variable.
PROJECT_DIR = Path(os.environ.get("DNA_BOOTCAMP_HOME", ".")).expanduser().resolve()
print("📁 Project directory:", PROJECT_DIR)
DATA_DIR = os.path.expanduser("/global/cfs/cdirs/m4388/projects/project7/ctcf_k562_example")
RESULTS_DIR = PROJECT_DIR / "notebook1_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Data directory :", DATA_DIR)
print("Results        :", RESULTS_DIR)
print("Device         :", device)

if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))

In [ ]:
# 🔒 RUN ONLY — load sequence and label files

seq_file = Path(DATA_DIR) / "seqs.txt"
label_file = Path(DATA_DIR) / "labels.txt"

print("Sequence file:", seq_file)
print("Label file   :", label_file)

if not seq_file.exists() or not label_file.exists():
    raise FileNotFoundError(
        f"Expected seqs.txt and labels.txt inside {DATA_DIR}"
    )

with open(seq_file) as f:
    raw_seqs = [
        line.strip().upper()
        for line in f
        if line.strip()
    ]

with open(label_file) as f:
    raw_labels = [
        int(line.strip())
        for line in f
        if line.strip()
    ]

assert len(raw_seqs) == len(raw_labels), (
    "The number of sequences and labels must match."
)

df = pd.DataFrame({
    "sequence": raw_seqs,
    "label": raw_labels,
})

df["label_name"] = df["label"].map({
    0: "Background",
    1: "Binding",
})

print(f"Loaded {len(df):,} rows.")

## Four pandas commands you will use repeatedly

```python
df.head()                       # show the first rows
df["label"].value_counts()      # count categories
df["length"].describe()         # summarize a numeric column
df.groupby("label_name")[...]   # compare groups
```

You do not need to memorize them immediately. You will use them several times.

In [ ]:
# 🔲 TRY IT — inspect the first five rows
df.head()

In [ ]:
# 🔲 TRY IT — basic questions about the table
print("Rows, columns:", df.shape)
print()
print("Class counts:")
print(df["label_name"].value_counts())

### ✏️ Explore a sequence yourself

Change `ROW_TO_VIEW` to another row number.

In [ ]:
# ✏️ EDIT ME
ROW_TO_VIEW = 0

row = df.iloc[ROW_TO_VIEW]

print("Row:", ROW_TO_VIEW)
print("Label:", row["label_name"])
print("DNA:")
print(row["sequence"])

# 3. Explore the Dataset Before Modeling

This is called **exploratory data analysis (EDA)**.

We are asking simple questions such as:

- Are the classes balanced?
- Are sequence lengths consistent?
- Do the sequences contain unexpected characters?
- Does GC content differ between classes?
- Are there repeated sequence patterns?

None of these questions requires a Transformer.

## 3A. Graph 1 — Class Balance

A **bar graph** is useful for comparing categories.

The code below has only three essential steps:

1. count each class,
2. draw bars,
3. label the graph.

In [ ]:
# 👀 READ — a simple bar graph
class_counts = df["label_name"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(class_counts.index, class_counts.values)
plt.xlabel("Class")
plt.ylabel("Number of sequences")
plt.title("Class Balance")
plt.tight_layout()
plt.show()

### 🔲 TRY IT

Using the output above:

- Which class has more examples?
- Is the difference large enough that class imbalance immediately worries you?
- Change the graph title to `"My First Genomics Plot"` and run it again.

## 3B. Graph 2 — Sequence Length

First create a new DataFrame column.

The Python expression:

```python
len(seq)
```

returns the number of characters in one sequence.

In [ ]:
# 👀 READ — create a numeric feature
df["length"] = df["sequence"].apply(len)

df["length"].describe()

In [ ]:
# 👀 READ — histogram of sequence lengths
plt.figure(figsize=(7, 4))
plt.hist(df["length"], bins=20)
plt.xlabel("Sequence length (bp)")
plt.ylabel("Number of sequences")
plt.title("Sequence Length Distribution")
plt.tight_layout()
plt.show()

### ✅ CHECKPOINT 2

What does the histogram tell you that simply printing one sequence length would not?

A histogram shows the **distribution across the entire dataset**, so unusual lengths are easier to notice.

## 3C. Graph 3 — GC Content

**GC content** is the fraction of a DNA sequence made of `G` or `C`.

This is also a small Python-function exercise.

-

### 🧩 Function spotlight — `gc_content`

This is a good first function to read because it has one clear job:

```text
DNA sequence → fraction of bases that are G or C
```

`gc_content(sequence)` does **not** change the sequence. It calculates one numeric feature and returns it.


In [ ]:
# 👀 READ — a reusable Python function
def gc_content(seq):
    n_gc = seq.count("G") + seq.count("C")
    return n_gc / len(seq)

# Apply the function to EVERY row.
df["gc_content"] = df["sequence"].apply(gc_content)

df[["sequence", "label_name", "gc_content"]].head()

In [ ]:
# 🔲 TRY IT — summarize GC content by class
df.groupby("label_name")["gc_content"].describe()

In [ ]:
# 👀 READ — box plot compares distributions between groups
binding_gc = df.loc[df["label"] == 1, "gc_content"]
background_gc = df.loc[df["label"] == 0, "gc_content"]

plt.figure(figsize=(6, 4))
plt.boxplot([background_gc, binding_gc], tick_labels=["Background", "Binding"]
)
plt.ylabel("GC content")
plt.title("GC Content by Class")
plt.tight_layout()
plt.show()

### 🔲 TRY IT

Do the two GC-content distributions look identical?

Even if they differ, **do not conclude that GC content causes CTCF binding**. This is exploratory evidence, not a causal experiment.

## 3D. Optional Exploration — Repeated DNA “Words”

A **k-mer** is a DNA substring of length `k`.

For example, with `k = 3`:

```text
ACGTG
ACG
 CGT
  GTG
```

Change `K_TO_EXPLORE` and see which short patterns are common in the binding sequences.

### 🧩 Function spotlight — `count_kmers`

A **k-mer** is a DNA “word” of length `k`.

```text
sequence = ACGTAC
k = 3
→ ACG, CGT, GTA, TAC
```

`count_kmers(...)` slides across many sequences and counts how often each DNA word occurs. `Counter` is just a Python object designed for counting repeated items.


In [ ]:
# ✏️ EDIT ME — try 2, 3, 4, or 6
K_TO_EXPLORE = 6

def count_kmers(sequences, k):
    counts = Counter()

    for seq in sequences:
        for i in range(len(seq) - k + 1):
            counts[seq[i:i+k]] += 1

    return counts

binding_sequences = df.loc[df["label"] == 1, "sequence"]
binding_kmers = count_kmers(binding_sequences, K_TO_EXPLORE)

top_kmers = pd.DataFrame(binding_kmers.most_common(10),
    columns=["kmer", "count"])

top_kmers

In [ ]:
# 👀 READ — graph your top k-mers
plt.figure(figsize=(8, 4))
plt.bar(top_kmers["kmer"], top_kmers["count"])
plt.xlabel(f"{K_TO_EXPLORE}-mer")
plt.ylabel("Count")
plt.title(f"Most Common {K_TO_EXPLORE}-mers in Binding Sequences")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

This graph is an **exploration tool**, not a binding-motif detector by itself. Common k-mers can be influenced by sequence composition and many other factors.

Later, DNABERT will learn representations of sequence patterns automatically.

# 4. Audit and Clean the Dataset

A powerful model will learn whatever pattern helps reduce its loss — including **mistakes or shortcuts in the dataset**.

Before training, we therefore check:

1. expected sequence length,
2. valid DNA characters,
3. duplicate sequences,
4. conflicting labels for the same sequence.

In [ ]:
# 👀 READ — add quality-control columns
expected_length = df["length"].mode().iloc[0]

df["length_ok"] = df["length"] == expected_length
df["valid_dna"] = df["sequence"].apply(lambda seq: set(seq) <= set("ACGT"))

# duplicated() checks whether the same DNA string appears more than once.
df["duplicate_sequence"] = df.duplicated(subset="sequence", keep=False)

print("Expected length:", expected_length)
print("Wrong length    :", (~df["length_ok"]).sum())
print("Invalid DNA     :", (~df["valid_dna"]).sum())
print("Rows involved in duplicate sequences:", df["duplicate_sequence"].sum())

In [ ]:
# 🔒 RUN ONLY — identify conflicting labels and create one clean sequence per row

labels_per_sequence = df.groupby("sequence")["label"].nunique()
conflicting_sequences = set(labels_per_sequence[labels_per_sequence > 1].index)

clean_df = df[df["length_ok"] & df["valid_dna"]
    & ~df["sequence"].isin(conflicting_sequences)].copy()

# Avoid identical DNA appearing in both train and validation later.
clean_df = clean_df.drop_duplicates(subset="sequence", keep="first"
).reset_index(drop=True)

print(f"Raw rows            : {len(df):,}")
print(f"Conflicting sequences: {len(conflicting_sequences):,}")
print(f"Clean unique rows   : {len(clean_df):,}")
print()
print(clean_df["label_name"].value_counts())

### ✅ CHECKPOINT 3

Why remove identical duplicate DNA sequences **before** making the train/validation split?

Because the same sequence appearing in both sets would let validation contain an example the model effectively already saw during training.

# 5. Meet DNABERT

DNABERT treats DNA as a sequence of genomic “words.”

The DNABERT model we use was pretrained with **6-mers**.

For a sequence such as:

```text
ACGTGCA
```

overlapping 6-mers are:

```text
ACGTGC
 CGTGCA
```

For a sequence of length `L`, overlapping k-mers produce:

```text
L - k + 1 tokens
```

So a 200-bp sequence gives **195 overlapping 6-mers** before special tokens are added.

```mermaid
flowchart LR
    A["Raw DNA"] --> B["Overlapping 6-mers"]
    B --> C["Vocabulary lookup"]
    C --> D["Token IDs"]
    D --> E["Embedding vectors"]
    E --> F["DNABERT Transformer"]
    F --> G["Context-aware vectors"]
```

In [ ]:
# 🔒 RUN ONLY — load DNABERT
from transformers import AutoTokenizer, AutoModel, AutoConfig

MODEL_NAME = "zhihan1996/DNA_bert_6"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)
base_model.eval()

def seq_to_kmer_sentence(seq, k=6):
    return " ".join(seq[i:i+k] for i in range(len(seq) - k + 1))

print("✅ DNABERT loaded")

## 5A. Follow One DNA Sequence Through the Tokenizer

In [ ]:
# ✏️ EDIT ME — choose another clean row
DEMO_INDEX = 0
DEMO_BASES = 30

demo_seq = clean_df.iloc[DEMO_INDEX]["sequence"][:DEMO_BASES]
demo_sentence = seq_to_kmer_sentence(demo_seq, k=6)

demo_tokens = tokenizer.tokenize(demo_sentence)
demo_ids = tokenizer.convert_tokens_to_ids(demo_tokens)

print("DNA:")
print(demo_seq)
print()
print("6-mers:")
print(demo_sentence.split()[:10])
print()
print("Tokenizer output:")
print(demo_tokens[:10])
print()
print("Token IDs:")
print(demo_ids[:10])

### Token vs. Token ID vs. Embedding

These are different things:

```mermaid
flowchart LR
    A["6-mer<br/>ACGTGC"] --> B["Token ID<br/>integer"]
    B --> C["Embedding lookup"]
    C --> D["768-number vector"]
```

- **Token** = DNA “word”
- **Token ID** = integer used to look up that token
- **Embedding** = learned numerical representation

In [ ]:
# 👀 READ — inspect one embedding
embedding_table = (base_model.embeddings.word_embeddings.weight .detach()
    .cpu())

example_token = demo_tokens[0]
example_id = tokenizer.convert_tokens_to_ids(example_token)
example_embedding = embedding_table[example_id]

print("Token           :", example_token)
print("Token ID        :", example_id)
print("Embedding shape :", tuple(example_embedding.shape))
print("First 8 values  :", example_embedding[:8].numpy().round(3))

## 5B. What Happens Inside the Transformer?

You will build these pieces yourself in Notebook 2. For now, understand the information flow.

For every token, self-attention creates:

- **Query (Q):** What am I looking for?
- **Key (K):** What do I offer?
- **Value (V):** What information do I pass along?

```mermaid
flowchart TD
    A["Token representation"] --> B["Query Q"]
    A --> C["Key K"]
    A --> D["Value V"]

    B --> E["Compare Q with Keys"]
    C --> E

    E --> F["Softmax<br/>attention weights"]
    F --> G["Weighted combination of Values"]
    D --> G

    G --> H["Context-aware representation"]
```

In [ ]:
# 👀 READ — inspect the model without printing thousands of lines
config = base_model.config

print("DNABERT architecture")
print("-" * 35)
print("Vocabulary size    :", config.vocab_size)
print("Hidden size        :", config.hidden_size)
print("Transformer layers :", config.num_hidden_layers)
print("Attention heads    :", config.num_attention_heads)

## 5C. From DNABERT to Binding / Background

DNABERT itself produces contextual representations. For classification, we add a small new **classification head**.

```mermaid
flowchart TD
    A["Tokenized DNA"] --> B["DNABERT"]
    B --> C["Final [CLS] vector<br/>768 numbers"]
    C --> D["Dropout"]
    D --> E["Linear layer<br/>768 → 2"]
    E --> F["Binding score"]
    E --> G["Background score"]
```

The new classifier begins with random weights. Fine-tuning teaches this head—and, depending on our choice, some or all of DNABERT—to solve our CTCF problem.

In [ ]:
# 🔒 RUN ONLY — one forward pass through pretrained DNABERT
demo_full_seq = clean_df.iloc[DEMO_INDEX]["sequence"]
demo_input = tokenizer(seq_to_kmer_sentence(demo_full_seq),
    return_tensors="pt", truncation=True, max_length=256)

with torch.no_grad():
    demo_output = base_model(**demo_input)

print("DNABERT output shape:", tuple(demo_output.last_hidden_state.shape))

cls_vector = demo_output.last_hidden_state[:, 0, :]
print("[CLS] shape:", tuple(cls_vector.shape))

### ✅ CHECKPOINT 4

Explain the pipeline in your own words:

**DNA → 6-mers → token IDs → embeddings → Transformer → [CLS] → classifier**

If you can explain what each arrow means, you understand the model at the level needed for fine-tuning.

# 6. Prepare Training and Validation Data

We need two different roles for data:

- **Training set:** updates model weights.
- **Validation set:** measures performance on examples that do not update the model.

We use a fixed random seed and `stratify` so every student starts with the same split and approximately the same class balance.

In [ ]:
# 🔒 RUN ONLY — fixed train/validation split
train_df, val_df = train_test_split(clean_df, test_size=0.20,
    random_state=RANDOM_SEED, stratify=clean_df["label"])

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training rows  :", len(train_df))
print("Validation rows:", len(val_df))

In [ ]:
# 👀 READ — another easy pandas graph
split_table = pd.DataFrame({"Training": train_df["label_name"].value_counts(),
    "Validation": val_df["label_name"].value_counts()}).fillna(0)

split_table.plot(kind="bar", figsize=(7, 4))
plt.ylabel("Number of sequences")
plt.title("Class Balance After Train/Validation Split")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## What Does “Training” Actually Do?

One batch follows this core pattern:

```python
optimizer.zero_grad()       # 1. clear old gradients
logits = model(...)         # 2. make predictions
loss = loss_function(...)   # 3. measure error
loss.backward()             # 4. calculate gradients
optimizer.step()            # 5. update trainable weights
```

This is the heart of fine-tuning.

Everything else in a training loop mainly repeats these steps across batches and epochs while recording metrics.

### ✅ CHECKPOINT 5

Which line actually changes the model's trainable parameters?

> `optimizer.step()`

`loss.backward()` calculates the gradients. `optimizer.step()` uses those gradients to update the parameters.

# 7. Fine-Tuning DNABERT — The Training Engine

Everything you need to run an experiment lives in the next five cells. They
were one very long cell; they are split up so you can find things.

| Cell | What it defines | Read it? |
|---|---|---|
| 7.1 | `EncodedDNADataset` — tokenize once, reuse | skim |
| 7.2 | `DNABertClassifier`, `configure_finetuning` | **yes** |
| 7.3 | `run_epoch` — the actual training loop | **yes** |
| 7.4 | `run_dnabert_experiment` — the runner you call | skim |
| 7.5 | plotting helpers | later |

Run all five now. Read 7.2 and 7.3 carefully — that is where the ideas from
Sections 5 and 6 turn into code.

### 7.1 — Package the data so the GPU can eat it

`EncodedDNADataset` runs the 6-mer tokenizer **once**, up front, instead of
re-tokenizing every sequence on every epoch. That is a pure speed decision:
the tokenizer is CPU work, and while the CPU tokenizes, the GPU sits idle.

Look for `padding="max_length"` — every sequence is padded to the same length
so a batch can be one rectangular tensor.

## 🧩 Read this before the training infrastructure

The next cells contain several reusable pieces. You are **not** expected to memorize their implementation.

### `EncodedDNADataset`

PyTorch wants a dataset object that knows three things:

| Method | Question it answers |
|---|---|
| `__init__` | What data should I store when the dataset is created? |
| `__len__` | How many examples are in the dataset? |
| `__getitem__(i)` | Give me example number `i`. |

### `DataLoader`

A `DataLoader` takes examples from a `Dataset` and groups them into **batches** for training.

```text
Dataset → individual examples → DataLoader → batch of examples → GPU
```

### `batch_size`

How many examples the model processes together before one optimizer update.


In [ ]:
# 🔒 RUN ONLY — reusable training infrastructure

class EncodedDNADataset(Dataset):
    """Pre-tokenize sequences once so repeated epochs spend less time tokenizing."""

    def __init__(self, seqs, labels, tokenizer, max_length=256):
        sentences = [seq_to_kmer_sentence(seq, 6) for seq in seqs]

        encoded = tokenizer(sentences, padding="max_length", truncation=True,
            max_length=max_length, return_tensors="pt")

        self.input_ids = encoded["input_ids"]
        self.attention_mask = encoded["attention_mask"]
        self.labels = torch.tensor(np.asarray(labels, dtype=np.int64),
            dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "label": self.labels[idx]}

### 7.2 — The model: pretrained DNABERT + a brand-new head

`DNABertClassifier` is the picture from Section 5C in code:
DNABERT produces a `[CLS]` vector, dropout regularizes it, one `Linear`
layer maps 768 numbers → 2 scores.

`configure_finetuning` implements the `head_only` / `last_4` / `full` choice
by setting `requires_grad` on each parameter. **This is the only thing that
changes between those three modes** — the architecture is identical.

## 🧩 Function map — the classifier

| Piece | Plain-English job |
|---|---|
| `DNABertClassifier.__init__` | Builds the model pieces: pretrained DNABERT + dropout + a new 2-class prediction layer. |
| `DNABertClassifier.forward` | Describes how one batch flows through those pieces to produce two scores per sequence. |
| `configure_finetuning` | Chooses which DNABERT parameters are allowed to change during training. |

**Logits:** the model's raw output scores *before* converting them into probabilities.

**Classifier head:** the small final layer that turns DNABERT's sequence representation into `Background` vs `Binding` scores.


In [ ]:
class DNABertClassifier(nn.Module):
    """Pretrained DNABERT + dropout + a new 2-class linear head."""

    def __init__(self, bert, hidden_size, dropout=0.1):
        super().__init__()
        self.bert = bert
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        cls_vector = output.last_hidden_state[:, 0, :]

        return self.classifier(self.dropout(cls_vector))


def configure_finetuning(bert, mode):
    """
    Choose how much of pretrained DNABERT is allowed to change.

    full       -> all DNABERT layers train
    last_4     -> only final 4 Transformer layers train
    head_only  -> DNABERT frozen; only new classifier trains
    """

    # Start by freezing the entire pretrained model.
    for param in bert.parameters():
        param.requires_grad = False

    if mode == "full":
        for param in bert.parameters():
            param.requires_grad = True

    elif mode == "last_4":
        for layer in bert.encoder.layer[-4:]:
            for param in layer.parameters():
                param.requires_grad = True

    elif mode == "head_only":
        pass

    else:
        raise ValueError(
            "FINETUNE_MODE must be 'full', 'last_4', or 'head_only'.")

### 7.3 — One epoch — the five lines you were promised

This is the block to actually read. Inside `run_epoch` you will find the
exact pattern from Section 6:

```python
optimizer.zero_grad()   # clear old gradients
logits = model(...)     # predict
loss = loss_fn(...)     # measure error
loss.backward()         # compute gradients
optimizer.step()        # update weights
```

Everything around those five lines is bookkeeping: moving tensors to the
GPU, accumulating the loss, and collecting probabilities for the metrics.

Note that validation runs under `torch.no_grad()` — no gradients, no weight
updates. That is what makes it a fair measurement.

## 🧩 Function map — one pass through the data

| Function | What it does |
|---|---|
| `run_epoch(...)` | Runs one complete pass through a DataLoader. In training mode it computes gradients and updates parameters; in validation mode it only measures performance. |
| `calculate_binary_metrics(...)` | Converts true labels, predicted labels, and probabilities into accuracy, precision, recall, specificity, F1, AUROC, AUPRC, and confusion counts. |

### Five lines to recognize during training

```python
optimizer.zero_grad()   # clear old gradients
logits = model(...)     # make predictions
loss = ...              # measure prediction error
loss.backward()         # calculate gradients
optimizer.step()        # update trainable parameters
```

**Gradient:** information telling a parameter which direction would reduce the loss.

**Optimizer:** the algorithm that uses gradients to update model parameters.


In [ ]:
def run_epoch(model, optimizer, loader, train=True, scheduler=None,
              use_amp=False):
    """
    Run one full pass through a DataLoader.

    During training:
        forward pass -> loss -> gradients -> clip -> update -> LR schedule

    During validation:
        forward pass -> metrics only; NO weight updates
    """

    model.train() if train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    all_scores = []
    all_predictions = []
    all_true = []

    context = torch.enable_grad() if train else torch.no_grad()

    with context:
        for batch in loader:
            ids = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            # bfloat16 on A100 tensor cores: same maths, roughly half the
            # time and half the memory. No GradScaler needed for bf16.
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16,
                                enabled=use_amp):
                logits = model(ids, mask)
                loss = F.cross_entropy(logits, labels)

            if train:
                loss.backward()

                # Clip gradients before the update. A single huge gradient
                # can undo a lot of pretraining in one step.
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

            logits = logits.float()
            predictions = logits.argmax(dim=1)

            # Probability assigned to the positive class: Binding = 1
            probabilities = torch.softmax(logits, dim=1)[:, 1]

            total_loss += loss.item() * labels.size(0)
            total_correct += (predictions == labels).sum().item()
            total_examples += labels.size(0)

            all_scores.extend(probabilities.detach().cpu().numpy())
            all_predictions.extend(predictions.detach().cpu().numpy())
            all_true.extend(labels.detach().cpu().numpy())

    return {"loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
        "scores": np.asarray(all_scores),
        "predictions": np.asarray(all_predictions),
        "true": np.asarray(all_true)}


def calculate_binary_metrics(true_labels, predicted_labels, scores):
    """
    Calculate a compact set of binary-classification metrics.

    Positive class = Binding (1)
    Negative class = Background (0)
    """

    tn, fp, fn, tp = confusion_matrix(true_labels, predicted_labels,
        labels=[0, 1]).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {"accuracy": (tp + tn) / (tp + tn + fp + fn),
        "precision": precision_score(true_labels, predicted_labels,
            zero_division=0), "recall": recall_score(true_labels,
            predicted_labels, zero_division=0), "specificity": specificity,
        "f1": f1_score(true_labels, predicted_labels, zero_division=0),
        "auroc": roc_auc_score(true_labels, scores),
        "auprc": average_precision_score(true_labels, scores), "tn": int(tn),
        "fp": int(fp), "fn": int(fn), "tp": int(tp)}

### 7.4 — The experiment runner

`run_dnabert_experiment` is the function you call from every experiment
cell. It builds the model, loops over epochs, records per-epoch history, and
writes one JSON record per run so you can compare runs later.

Every run is saved. That is what makes Section 11 possible.

## 🧩 Function map — experiment runner

This cell is longer because it automates a repeatable experiment.

| Function | Plain-English job |
|---|---|
| `save_experiment_record` | Appends one run's summary to the experiment CSV. |
| `make_linear_schedule` | Creates a learning-rate schedule that changes gradually during training. |
| `unwrap` | If a model is wrapped for multiple GPUs, returns the underlying model. Otherwise returns the model unchanged. |
| `run_dnabert_experiment` | The main coordinator: creates data loaders/model/optimizer, trains epochs, validates, saves results, and returns useful objects. |
| `load_notebook1_results` | Reads saved experiment summaries back into a DataFrame. |

**Scheduler:** a rule for changing the learning rate as training progresses.

When reading `run_dnabert_experiment`, focus on the **order of operations**, not every line.


In [ ]:
def save_experiment_record(record, history_df, predictions_df):
    """Save summary, epoch history, and final validation predictions."""

    csv_path = RESULTS_DIR / "notebook1_experiments.csv"

    if csv_path.exists():
        results_df = pd.read_csv(csv_path)

        # Re-running the same name replaces the old row.
        results_df = results_df[results_df["run_name"] != record["run_name"]]

        results_df = pd.concat([results_df, pd.DataFrame([record])],
            ignore_index=True)
    else:
        results_df = pd.DataFrame([record])

    results_df.to_csv(csv_path, index=False)

    history_path = (RESULTS_DIR / f"{record['run_name']}_history.csv")
    history_df.to_csv(history_path, index=False)

    predictions_path = (RESULTS_DIR / f"{record['run_name']}_predictions.csv")
    predictions_df.to_csv(predictions_path, index=False)

    summary_path = (RESULTS_DIR / f"{record['run_name']}_summary.json")

    with open(summary_path, "w") as f:
        json.dump(record, f, indent=2)

    return (csv_path, history_path, predictions_path, summary_path)


def make_linear_schedule(optimizer, total_steps, warmup_ratio=0.1):
    """Warm the learning rate up, then decay it linearly to zero.

    Starting a pretrained model at full learning rate on step 1 is one of
    the reliable ways to destroy it. Warmup eases in.
    """
    warmup_steps = max(1, int(total_steps * warmup_ratio))

    def lr_factor(step):
        if step < warmup_steps:
            return step / warmup_steps
        remaining = max(0, total_steps - step)
        return remaining / max(1, total_steps - warmup_steps)

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)


def unwrap(model):
    """Return the real model, whether or not DataParallel wrapped it."""
    return model.module if isinstance(model, nn.DataParallel) else model


def run_dnabert_experiment(run_name, epochs, batch_size, learning_rate,
    dropout, finetune_mode, max_length=200, random_seed=42,
    use_all_gpus=False, num_workers=4, weight_decay=0.01,
    warmup_ratio=0.1):
    """Train one fresh DNABERT classifier and save its results."""

    torch.manual_seed(random_seed)
    np.random.seed(random_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_seed)

    train_dataset = EncodedDNADataset(train_df["sequence"].tolist(),
        train_df["label"].tolist(), tokenizer, max_length=max_length)

    val_dataset = EncodedDNADataset(val_df["sequence"].tolist(),
        val_df["label"].tolist(), tokenizer, max_length=max_length)

    generator = torch.Generator()
    generator.manual_seed(random_seed)

    # num_workers > 0 lets the CPU prepare the next batch while the GPU is
    # busy with the current one.
    loader_kwargs = {"num_workers": num_workers, "pin_memory": True,
                     "persistent_workers": num_workers > 0}

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
        shuffle=True, generator=generator, drop_last=True, **loader_kwargs)

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
        **loader_kwargs)

    fresh_config = AutoConfig.from_pretrained(MODEL_NAME)

    fresh_bert = AutoModel.from_pretrained(MODEL_NAME, config=fresh_config)

    configure_finetuning(fresh_bert, finetune_mode)

    model = DNABertClassifier(fresh_bert, fresh_config.hidden_size,
        dropout=dropout).to(device)

    n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0

    if use_all_gpus and n_gpus > 1:
        model = nn.DataParallel(model)

    active_gpus = n_gpus if (use_all_gpus and n_gpus > 1) else min(n_gpus, 1)

    # AdamW applies weight decay correctly for transformers; plain Adam
    # folds it into the gradient, which is not the same thing.
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad,
            model.parameters()), lr=learning_rate, weight_decay=weight_decay)

    total_steps = len(train_loader) * epochs
    scheduler = make_linear_schedule(optimizer, total_steps, warmup_ratio)

    use_amp = torch.cuda.is_available()

    total_params = sum(p.numel() for p in model.parameters())

    trainable_params = sum(p.numel() for p in model.parameters()
        if p.requires_grad)

    history = []

    print("=" * 76)
    print("RUN:", run_name)
    print(f"Device: {device}  |  GPUs in use: {active_gpus}"
          f"{'  (DataParallel)' if active_gpus > 1 else ''}")
    print(f"epochs={epochs} | batch={batch_size} | "
        f"lr={learning_rate} | dropout={dropout} | max_length={max_length}")
    print(f"precision: {'bfloat16 (autocast)' if use_amp else 'float32'}")
    print(f"optimizer steps: {total_steps:,} "
          f"({len(train_loader):,} per epoch, "
          f"{int(total_steps * warmup_ratio):,} warmup)")
    print("fine-tune mode:", finetune_mode)
    print(f"trainable parameters: {trainable_params:,} / {total_params:,}")
    print("=" * 76)

    training_start = time.time()

    best_auroc = -1.0
    best_state = None
    best_val_metrics = None

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()

        train_metrics = run_epoch(model, optimizer, train_loader, train=True,
            scheduler=scheduler, use_amp=use_amp)

        val_metrics = run_epoch(model, optimizer, val_loader, train=False,
            use_amp=use_amp)

        val_auroc = roc_auc_score(val_metrics["true"], val_metrics["scores"])

        val_auprc = average_precision_score(val_metrics["true"],
            val_metrics["scores"])

        epoch_seconds = time.time() - epoch_start
        n_seen = len(train_dataset) + len(val_dataset)

        history.append({"epoch": epoch, "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"], "val_auroc": val_auroc,
            "val_auprc": val_auprc, "epoch_seconds": epoch_seconds,
            "examples_per_second": n_seen / epoch_seconds})

        # Keep the BEST model, not the last one. Without this, one bad epoch
        # at the end throws away everything the earlier epochs learned.
        if val_auroc > best_auroc:
            best_auroc = val_auroc
            best_val_metrics = val_metrics
            best_state = {k: v.detach().cpu().clone()
                          for k, v in unwrap(model).state_dict().items()}
            marker = "  <- best so far"
        else:
            marker = ""

        print(f"Epoch {epoch}/{epochs} | "
            f"train loss={train_metrics['loss']:.4f} | "
            f"train acc={train_metrics['accuracy']:.3f} | "
            f"val loss={val_metrics['loss']:.4f} | "
            f"val acc={val_metrics['accuracy']:.3f} | "
            f"AUROC={val_auroc:.4f} | AUPRC={val_auprc:.4f} | "
            f"{epoch_seconds:.1f}s{marker}")

        # Divergence guard: ln(2) = 0.693 is the loss of a model that has
        # given up and outputs 50/50 on everything.
        if val_metrics["loss"] > 0.69 and val_auroc < 0.55:
            print()
            print("  🚨 This epoch collapsed to chance "
                  "(val loss ≈ ln(2), AUROC ≈ 0.5).")
            print("     The learning rate is almost certainly too high for "
                  "this many steps.")
            print("     Lower LEARNING_RATE (try 2e-5) and re-run.")
            print()

    total_training_time = (time.time() - training_start)

    # Restore the best epoch's weights before reporting or predicting.
    if best_state is not None:
        unwrap(model).load_state_dict(best_state)

    final_val_metrics = best_val_metrics

    history_df = pd.DataFrame(history)

    best_row = history_df.loc[history_df["val_auroc"].idxmax()]

    print()
    print(f"↩️  Restored weights from epoch {int(best_row['epoch'])} "
          f"(best AUROC). Metrics below describe THAT model.")

    final_class_metrics = calculate_binary_metrics(final_val_metrics["true"],
        final_val_metrics["predictions"], final_val_metrics["scores"])

    predictions_df = val_df[["sequence", "label", "label_name"]].copy()

    predictions_df["predicted_label"] = (final_val_metrics["predictions"])

    predictions_df["predicted_name"] = (predictions_df["predicted_label"].map({
            0: "Background", 1: "Binding"}))

    predictions_df["binding_probability"] = (final_val_metrics["scores"])

    predictions_df["correct"] = (predictions_df["label"]
        == predictions_df["predicted_label"])

    record = {"run_name": run_name, "model": "DNABERT-6",
        "device": str(device), "epochs": int(epochs),
        "batch_size": int(batch_size), "learning_rate": float(learning_rate),
        "max_length": int(max_length), "dropout": float(dropout),
        "finetune_mode": finetune_mode, "random_seed": int(random_seed),
 "trainable_parameters": int(trainable_params),
        "total_parameters": int(total_params),
 "best_val_auroc": float(best_row["val_auroc"]),
        "best_epoch": int(best_row["epoch"]),
 "final_val_loss": float(history_df.iloc[-1]["val_loss"]),
        "final_val_accuracy": float(final_class_metrics["accuracy"]),
        "final_val_precision": float(final_class_metrics["precision"]),
        "final_val_recall": float(final_class_metrics["recall"]),
        "final_val_specificity": float(final_class_metrics["specificity"]),
        "final_val_f1": float(final_class_metrics["f1"]),
        "final_val_auroc": float(final_class_metrics["auroc"]),
        "final_val_auprc": float(final_class_metrics["auprc"]),
 "true_negative": int(final_class_metrics["tn"]), "false_positive": int(
            final_class_metrics["fp"]), "false_negative": int(
            final_class_metrics["fn"]), "true_positive": int(
            final_class_metrics["tp"]),
 "total_training_time_seconds": float(total_training_time)}

    save_experiment_record(record, history_df, predictions_df)

    print()
    print("FINAL VALIDATION METRICS")
    print("-" * 34)
    print(f"Accuracy    : " f"{record['final_val_accuracy']:.4f}")
    print(f"Precision   : " f"{record['final_val_precision']:.4f}")
    print(f"Recall      : " f"{record['final_val_recall']:.4f}")
    print(f"Specificity : " f"{record['final_val_specificity']:.4f}")
    print(f"F1 score    : " f"{record['final_val_f1']:.4f}")
    print(f"AUROC       : " f"{record['final_val_auroc']:.4f}")
    print(f"AUPRC       : " f"{record['final_val_auprc']:.4f}")
    print()
    print(f"✅ Best AUROC: " f"{record['best_val_auroc']:.4f} "
        f"(epoch {record['best_epoch']})")
    print(f"⏱️ Training time: " f"{total_training_time:.1f}s")
    print("💾 Summary, history, and predictions saved.")

    return (model, history_df, record, predictions_df)


def load_notebook1_results():
    path = RESULTS_DIR / "notebook1_experiments.csv"

    if not path.exists():
        return pd.DataFrame()

    return pd.read_csv(path)

### 7.5 — Evaluation plots

Three plotting helpers used from Section 9C onward. Each takes the
predictions table and draws one figure. You do not need to read these now —
come back when you want to change how a figure looks.

## 🧩 Plot helper functions

These functions do not train anything. They take already-saved predictions and turn them into standard diagnostic plots:

- `plot_confusion_matrix_from_predictions` → counts TN / FP / FN / TP
- `plot_roc_from_predictions` → ROC curve
- `plot_precision_recall_from_predictions` → precision–recall curve

A plotting helper exists so we can reuse the same graph code for every experiment.


In [ ]:
def plot_confusion_matrix_from_predictions(predictions_df):
    """Simple confusion-matrix plot without extra plotting libraries."""

    cm = confusion_matrix(predictions_df["label"],
        predictions_df["predicted_label"], labels=[0, 1])

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.xticks([0, 1], ["Background", "Binding"])
    plt.yticks([0, 1], ["Background", "Binding"])
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title("Confusion Matrix")

    for row in range(2):
        for col in range(2):
            plt.text(col, row, cm[row, col], ha="center", va="center")

    plt.colorbar(label="Number of sequences")
    plt.tight_layout()
    plt.show()


def plot_roc_from_predictions(predictions_df):
    fpr, tpr, _ = roc_curve(predictions_df["label"],
        predictions_df["binding_probability"])

    auroc = roc_auc_score(predictions_df["label"],
        predictions_df["binding_probability"])

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"Model (AUROC = {auroc:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Random ranking")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate / Recall")
    plt.title("ROC Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_precision_recall_from_predictions(predictions_df):
    precision, recall, _ = precision_recall_curve(predictions_df["label"],
        predictions_df["binding_probability"])

    auprc = average_precision_score(predictions_df["label"],
        predictions_df["binding_probability"])

    baseline_fraction = predictions_df["label"].mean()

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"Model (AUPRC = {auprc:.3f})")
    plt.axhline(baseline_fraction, linestyle="--",
        label=f"Positive fraction = {baseline_fraction:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision–Recall Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()


print("✅ Fine-tuning and evaluation infrastructure ready.")

# 8. Run a Baseline

A **baseline** gives us something to compare future experiments against.

For the baseline, we use:

```text
epochs        = 2
batch size    = 8
learning rate = 1e-4
dropout       = 0.1
fine-tuning   = full DNABERT
```

We keep `max_length = 256` because our 200-bp windows already fit after 6-mer tokenization.

## 7a. The Two Settings That Decide Whether This Works

Most hyperparameters are worth a shrug. Two are not.

### Learning rate — the one that can silently destroy the model

DNABERT arrives already knowing what human DNA looks like. Fine-tuning is
supposed to *adjust* that knowledge, not overwrite it. Too large a learning
rate overwrites it.

The standard range for full BERT fine-tuning is **2e-5 to 5e-5**. Here is
what a run at 1e-4 on this dataset looked like:

```text
Epoch 1/2 | train acc=0.848 | val acc=0.855 | AUROC=0.9417
Epoch 2/2 | train acc=0.736 | val acc=0.502 | AUROC=0.5063
```

Epoch 1 was excellent. Epoch 2 destroyed it. Note that **training** accuracy
fell too — that is the giveaway. Overfitting makes train accuracy climb while
validation falls. Both falling means the model itself is breaking apart.

The final validation loss was **0.6933**, and `ln(2) = 0.6931`. That is
precisely the loss of a model that outputs 50/50 on every input. It had
stopped predicting anything at all.

The catch: this same learning rate looked fine on a small dataset, because

```text
 1,440 training rows / batch 8  =    180 weight updates per epoch
79,321 training rows / batch 8  =  9,916 weight updates per epoch
```

**A learning rate that survives 180 updates can still explode over 9,916.**
The setting did not change; the number of chances it had to go wrong did.

Three defences are now built into the training engine:

| Defence | What it does |
|---|---|
| **Warmup** | starts the LR near zero and eases it in over the first 10% of steps |
| **Gradient clipping** | caps any single update, so one bad batch cannot wreck the weights |
| **Best-epoch checkpointing** | keeps the best model, so a late bad epoch cannot throw away a good one |

That last one also fixes a reporting bug: metrics used to describe the
*final* epoch even when an earlier one was better.

### Batch size — the one that decides whether the GPU is working

At `batch_size=8`, the run above managed about **116 examples/second** on an
A100, and 14 minutes per epoch. The GPU spent most of that time waiting for
work. Larger batches, bfloat16, and background data loading address it —
you will measure the difference yourself in 7c.

## 7b. Using More Than One GPU

Your first real experience of "this would be faster with more hardware."

One line does it:

```python
model = nn.DataParallel(model)
```

`DataParallel` splits each batch across the GPUs it can see, runs the forward
and backward pass on all of them at once, and sums the gradients. A batch of
64 on 4 GPUs means each GPU handles 16.

Note what does *not* change: the maths, the learning rate, the number of
optimizer steps. You are doing the same training, just faster. That is the
whole promise of parallelism, and the reason a scaling plot is meaningful at
all — the work has to be identical for a speed comparison to mean anything.

### How to actually get more GPUs

In Jupyter on Perlmutter, your notebook sees whatever the job that started it
requested. To get more, start the notebook from a job that asks for more:

```bash
# 2 GPUs on a shared node — fine for testing
salloc -A m4388 -C gpu -q shared -t 02:00:00 -n 2 --gpus-per-task=1

# a full node, 4 GPUs — during the bootcamp, add:
#   -q regular --reservation=bootcamp_day1
salloc -A m4388 -C gpu -q regular -N 1 --gpus-per-node=4 -t 02:00:00
```

Then re-run this notebook and the cell above will report more than one GPU.

### Why Notebook 3 does it differently

`DataParallel` is one line, but it has a real ceiling: a single Python
process drives every GPU, so one CPU thread becomes the bottleneck, and it
cannot cross node boundaries at all. Four GPUs is roughly where it stops
paying off.

**DistributedDataParallel (DDP)** runs one process per GPU and can span many
nodes — that is how Notebook 3A reaches 8, 32 or 160 GPUs. It cannot run
inside a notebook cell, which is why 3A submits a Slurm job instead.

So: `DataParallel` here to feel the effect, DDP there to scale it.

In [ ]:
# ✏️ BASELINE CONFIGURATION
#
# These values matter more than they look. See the section above for why
# LEARNING_RATE in particular is not a free choice.

USE_ALL_GPUS = True          # use every GPU the notebook can see

BASELINE_CONFIG = {
    "run_name": "baseline",
    "epochs": 3,

    # 2e-5 is the standard range for full BERT fine-tuning (2e-5 to 5e-5).
    # 1e-4 diverges on a dataset this size.
    "learning_rate": 2e-5,

    # Big batches keep the GPU busy. 8 leaves an A100 mostly idle.
    "batch_size": 64,

    # 200 bp -> 195 six-mers + [CLS] + [SEP] = 197 tokens. Padding to 256
    # would waste 23% of every attention computation on nothing.
    "max_length": 200,

    "dropout": 0.1,
    "finetune_mode": "full",
    "random_seed": RANDOM_SEED,
    "use_all_gpus": USE_ALL_GPUS,
}

visible_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"GPUs visible to this notebook: {visible_gpus}")

if USE_ALL_GPUS and visible_gpus > 1:
    print(f"  -> training will use all {visible_gpus} via DataParallel")
    print(f"  -> each GPU gets {BASELINE_CONFIG['batch_size'] // visible_gpus}"
          f" of every {BASELINE_CONFIG['batch_size']}-example batch")
elif visible_gpus == 1:
    print("  -> one GPU. Section 7b explains how to ask for more.")

BASELINE_CONFIG


In [ ]:
# ✏️ RUN THIS CELL when you are ready to train the baseline
baseline_model, baseline_history, baseline_result, baseline_predictions = (
    run_dnabert_experiment(**BASELINE_CONFIG))

In [ ]:
# 👀 READ — what did that run cost, and what would more GPUs buy?

epoch_seconds = baseline_history["epoch_seconds"].mean()
throughput = baseline_history["examples_per_second"].mean()
gpus_used = max(1, torch.cuda.device_count() if USE_ALL_GPUS else 1)

print("COST OF THE BASELINE RUN")
print("-" * 52)
print(f"  GPUs used            : {gpus_used}")
print(f"  Seconds per epoch    : {epoch_seconds:.1f}")
print(f"  Examples per second  : {throughput:,.0f}")
print(f"  Total training time  : "
      f"{baseline_result['total_training_time_seconds'] / 60:.1f} min")
print()

# Section 10 asks you to compare several configurations.
N_CONFIGS = 4
EPOCHS_EACH = BASELINE_CONFIG["epochs"]

serial_hours = (N_CONFIGS * EPOCHS_EACH * epoch_seconds) / 3600

print(f"NOW SCALE IT UP: {N_CONFIGS} configurations x "
      f"{EPOCHS_EACH} epochs")
print("-" * 52)
print(f"  At this speed        : {serial_hours:.1f} hours")
print()
print("  If scaling were perfect (it is not — you will measure the gap"
      " in 3A):")
for n in [4, 8, 16, 32]:
    print(f"    {n:>3} GPUs : {serial_hours * gpus_used / n:>5.2f} hours")
print()
print("This is why Notebook 3 exists. Not because bigger is impressive,")
print("but because the experiment you actually want to run does not fit")
print("in an afternoon on one GPU.")

## 7c. Where This Goes Next

You now have a working single-GPU (or few-GPU) fine-tuning run and a number
for what it costs. That number is the baseline every later comparison is
measured against — write it down.

**Notebook 2** opens the model up: you build the Transformer yourself, from
scratch, and find out how much of DNABERT's advantage is architecture and how
much is pretraining.

**Notebook 3A** takes exactly this fine-tuning job to many GPUs with DDP and
Slurm, measures the real speedup against the ideal, and then spends the
hardware on a maximum-power run.

**Notebook 3B** does the same for your from-scratch model, and asks whether it
can catch DNABERT.

The `total_training_time_seconds` in your saved summary is the first point on
the scaling curve you will build in 3A.

# 9. Read the Training Results

Two important quantities are:

### Loss
The error signal the optimizer tries to reduce.

### Validation AUROC
Measures how well the model ranks binding examples above background examples.

A rough interpretation:

- `0.5` ≈ random ranking
- closer to `1.0` = better separation

Do not judge a model from one number alone. Look at the **training pattern** too.

In [ ]:
# 👀 READ — Graph 1: training vs validation loss
plt.figure(figsize=(7, 4))

plt.plot(baseline_history["epoch"], baseline_history["train_loss"], marker="o",
    label="Training")

plt.plot(baseline_history["epoch"], baseline_history["val_loss"], marker="o",
    label="Validation")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DNABERT Baseline: Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 👀 READ — Graph 2: validation AUROC across epochs
plt.figure(figsize=(7, 4))

plt.plot(baseline_history["epoch"], baseline_history["val_auroc"], marker="o")

plt.axhline(0.5, linestyle="--", label="Random ranking")

plt.xlabel("Epoch")
plt.ylabel("Validation AUROC")
plt.title("DNABERT Baseline: AUROC")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

### ✅ CHECKPOINT 6

Look at your actual graphs.

1. Did training loss decrease?
2. Did validation loss decrease?
3. Did validation AUROC improve?
4. What was the best validation AUROC?
5. If training loss continues decreasing while validation loss gets worse, what might that suggest?

That last pattern is a common sign of **overfitting**.

# 9B. Evaluate More Than One Metric

A model can look good according to one metric and weaker according to another.

For a binary classifier, we will inspect:

| Metric | Simple question it answers |
|---|---|
| **Loss** | How wrong/confident are the model's probability predictions? |
| **Accuracy** | What fraction of examples got the correct class? |
| **Precision** | When the model says **Binding**, how often is it right? |
| **Recall / Sensitivity** | Of the real Binding sequences, how many did it find? |
| **Specificity** | Of the real Background sequences, how many did it correctly reject? |
| **F1 score** | How well are precision and recall balanced? |
| **AUROC** | How well does the model rank Binding above Background across thresholds? |
| **AUPRC** | How strong is the precision–recall tradeoff across thresholds? |

No single metric answers every question.

## Accuracy — Easy to Understand, But Not Always Enough

\[
\text{Accuracy} =
\frac{\text{correct predictions}}{\text{all predictions}}
\]

If 90 out of 100 sequences are classified correctly:

```text
accuracy = 90 / 100 = 0.90
```

Accuracy is useful here because our classes are fairly balanced.

But imagine a dataset with:

```text
990 Background
10 Binding
```

A model that predicts **Background every time** would have 99% accuracy while completely failing to detect Binding.

That is why we inspect additional metrics.

In [ ]:
# 👀 READ — Graph 3: training vs validation accuracy
plt.figure(figsize=(7, 4))

plt.plot(baseline_history["epoch"], baseline_history["train_accuracy"],
    marker="o", label="Training")

plt.plot(baseline_history["epoch"], baseline_history["val_accuracy"],
    marker="o", label="Validation")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("DNABERT Baseline: Accuracy")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

### What should you look for?

- If **training and validation accuracy both improve**, the model is learning useful patterns.
- If **training accuracy keeps rising but validation accuracy stops improving**, the model may be overfitting.
- A small gap is normal; a very large gap deserves attention.

Always compare this graph with the **loss graph**.

# 9C. Confusion Matrix — What Kinds of Mistakes Did the Model Make?

The confusion matrix breaks predictions into four groups:

| | Predicted Background | Predicted Binding |
|---|---:|---:|
| **True Background** | True Negative (TN) | False Positive (FP) |
| **True Binding** | False Negative (FN) | True Positive (TP) |

### For our biological question

**True Positive**
> Real CTCF-binding sequence predicted as Binding.

**True Negative**
> Background sequence predicted as Background.

**False Positive**
> Background sequence incorrectly predicted as Binding.

**False Negative**
> Real CTCF-binding sequence missed by the model.

The confusion matrix is especially useful because two models with similar accuracy can make **different kinds of mistakes**.

In [ ]:
# 👀 READ — confusion matrix
plot_confusion_matrix_from_predictions(baseline_predictions)

In [ ]:
# 🔲 TRY IT — see the actual four numbers
cm = confusion_matrix(baseline_predictions["label"],
    baseline_predictions["predicted_label"], labels=[0, 1])

tn, fp, fn, tp = cm.ravel()

print("True negatives :", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives :", tp)

# 9D. Precision, Recall, Specificity, and F1

These metrics use the same four confusion-matrix counts.

## Precision

> **When the model predicts Binding, how often is it correct?**

\[
\text{Precision} =
\frac{TP}{TP + FP}
\]

Low precision means the model produces many **false-positive Binding calls**.

---

## Recall / Sensitivity

> **Of all real Binding sequences, how many did the model find?**

\[
\text{Recall} =
\frac{TP}{TP + FN}
\]

Low recall means the model misses many real Binding sequences.

---

## Specificity

> **Of all real Background sequences, how many did the model correctly reject?**

\[
\text{Specificity} =
\frac{TN}{TN + FP}
\]

---

## F1 Score

F1 combines precision and recall into one score.

A high F1 requires both to be reasonably strong.

This is useful when you care about **finding positives without producing too many false positives**.

In [ ]:
# 👀 READ — compact metric table
baseline_metric_table = pd.DataFrame({"Metric": ["Accuracy", "Precision",
        "Recall / Sensitivity", "Specificity", "F1", "AUROC", "AUPRC"],
    "Value": [baseline_result["final_val_accuracy"],
        baseline_result["final_val_precision"],
        baseline_result["final_val_recall"],
        baseline_result["final_val_specificity"],
        baseline_result["final_val_f1"], baseline_result["final_val_auroc"],
        baseline_result["final_val_auprc"]]})

baseline_metric_table

In [ ]:
# 👀 READ — graph the final metrics
plt.figure(figsize=(8, 4))

plt.bar(baseline_metric_table["Metric"], baseline_metric_table["Value"])

plt.ylabel("Score")
plt.title("DNABERT Baseline: Validation Metrics")
plt.ylim(0, 1)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# 9E. ROC Curve — Performance Across Decision Thresholds

The classifier produces a **Binding probability**.

A default decision might be:

```text
probability >= 0.50 → Binding
probability < 0.50  → Background
```

But 0.50 is only one possible threshold.

The **ROC curve** asks:

> What happens to true positives and false positives as we move the threshold?

Axes:

- **x-axis:** False Positive Rate
- **y-axis:** True Positive Rate (Recall)

A model whose curve stays closer to the upper-left performs better at ranking the two classes.

### AUROC

**Area Under the ROC Curve** summarizes the whole ROC curve.

Rough intuition:

```text
0.50 ≈ random ranking
1.00 = perfect ranking
```

AUROC is about **ranking across thresholds**, not accuracy at one chosen threshold.

In [ ]:
# 👀 READ — ROC curve
plot_roc_from_predictions(baseline_predictions)

# 9F. Precision–Recall Curve

The **precision–recall curve** focuses on the positive class: **Binding**.

It asks how:

- **Precision** changes
- as **Recall** changes

when we move the classification threshold.

This can be particularly useful when the positive class is rare.

### AUPRC

AUPRC summarizes the precision–recall curve.

A larger value generally indicates a better precision/recall tradeoff.

Unlike AUROC, the baseline for a PR curve depends on the fraction of positive examples in the dataset.

In [ ]:
# 👀 READ — Precision–Recall curve
plot_precision_recall_from_predictions(baseline_predictions)

# 9G. Look at Individual Predictions

Metrics summarize hundreds of examples into numbers.

Sometimes it is useful to inspect actual predictions.

The saved table contains:

- true label,
- predicted label,
- probability of Binding,
- whether the prediction was correct.

In [ ]:
# 🔲 TRY IT — most confident Binding predictions
baseline_predictions.sort_values("binding_probability", ascending=False
).head(10)

In [ ]:
# 🔲 TRY IT — inspect mistakes only
baseline_predictions[baseline_predictions["correct"] == False].sort_values(
    "binding_probability", ascending=False).head(10)

# 9H. What Did the Model Actually Learn?

You have metrics. Now for the harder question — the one that separates
running a model from understanding one:

> **Is DNABERT finding real CTCF sequence patterns, or did it find an easier
> shortcut?**

Back in Section 3C you noticed that binding sequences look GC-richer than
background. A Transformer is perfectly capable of noticing that too — and if
GC content alone explains most of the performance, then all this machinery
bought you very little.

So let's test it. The honest way to find out is to build the dumbest possible
classifier and see how close it gets.

In [ ]:
# 👀 READ — the dumbest possible classifier: GC content alone

baseline_predictions["gc_content"] = (
    baseline_predictions["sequence"].apply(gc_content)
)

# "Predict Binding when GC content is high." No training, no parameters.
gc_only_auroc = roc_auc_score(
    baseline_predictions["label"],
    baseline_predictions["gc_content"],
)

dnabert_auroc = roc_auc_score(
    baseline_predictions["label"],
    baseline_predictions["binding_probability"],
)

print("Validation AUROC")
print("-" * 46)
print(f"  GC content alone (0 parameters) : {gc_only_auroc:.4f}")
print(f"  Fine-tuned DNABERT              : {dnabert_auroc:.4f}")
print(f"  Random guessing                 : 0.5000")
print()
print(f"  DNABERT beats the GC baseline by "
      f"{dnabert_auroc - gc_only_auroc:+.4f} AUROC")

In [ ]:
# 👀 READ — is the model's confidence just tracking GC content?

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for label, name in [(0, "Background"), (1, "Binding")]:
    subset = baseline_predictions[baseline_predictions["label"] == label]
    axes[0].scatter(subset["gc_content"], subset["binding_probability"],
                    alpha=0.4, s=14, label=name)

axes[0].set_xlabel("GC content")
axes[0].set_ylabel("Model's binding probability")
axes[0].set_title("Model confidence vs. GC content")
axes[0].axhline(0.5, linestyle=":", color="grey")
axes[0].legend()

correlation = baseline_predictions["gc_content"].corr(
    baseline_predictions["binding_probability"]
)

# Where does the model go wrong?
correct = baseline_predictions[baseline_predictions["correct"]]
wrong = baseline_predictions[~baseline_predictions["correct"]]

axes[1].boxplot([correct["gc_content"], wrong["gc_content"]],
                tick_labels=[f"Correct\n(n={len(correct)})",
                             f"Wrong\n(n={len(wrong)})"])
axes[1].set_ylabel("GC content")
axes[1].set_title("GC content of correct vs. incorrect predictions")

plt.tight_layout()
plt.show()

print(f"Correlation between GC content and binding probability: "
      f"{correlation:.3f}")

### ✅ CHECKPOINT 7 — Interpret your own numbers

1. **How much better is DNABERT than GC content alone?** If the gap is
   large, DNABERT is using something beyond base composition. If the gap is
   small, most of your AUROC was available for free.
2. **Look at the scatter plot.** If the points formed a tight rising line,
   the model would essentially *be* a GC detector. Does yours?
3. **Look at the box plot.** Do the model's mistakes sit at unusual GC
   values — background that happens to be GC-rich, binding sites that happen
   not to be?

### Why this section exists

Comparing against a trivial baseline is the habit worth taking out of this
notebook. A model that scores 0.95 sounds excellent right up until you learn
that counting `G` and `C` scores 0.90 — at which point you have spent a GPU
to buy 0.05.

This is not a reason to distrust the model. It is how you find out **what
you actually bought**, which is a question nobody can answer from the AUROC
number alone.

### 🧪 Try it yourself

The `predictions_df` table has the DNA sequence for every validation
example. Sort by `binding_probability` and look at the top 10 sequences the
model was most confident about. Do you see a repeated pattern by eye? CTCF's
motif is roughly `CCGCGNGGNGGCAG` — search a few of those sequences for
`CCGCG` or `GGCAG` and see whether the confident calls contain it more often
than the unconfident ones.

### ✅ CHECKPOINT — Choosing the Right Metric

For each question, which metric/plot would you inspect first?

1. **“How many total predictions were correct?”** → Accuracy
2. **“When the model says Binding, can I trust it?”** → Precision
3. **“How many real Binding examples did we find?”** → Recall
4. **“What kinds of mistakes are happening?”** → Confusion matrix
5. **“How well does the model rank the two classes across thresholds?”** → ROC / AUROC
6. **“How does precision trade off with recall?”** → PR curve / AUPRC
7. **“Is the model overfitting as epochs increase?”** → Training vs validation curves

# 10. Understanding Fine-Tuning Parameters

Before changing numbers, understand **what each parameter controls**.

There are two useful categories:

### Training hyperparameters
These control **how the optimizer learns**.

- `epochs`
- `batch_size`
- `learning_rate`

### Model / regularization choices
These affect **what can change or how strongly the model is regularized**.

- `dropout`
- `finetune_mode`
- `max_length`

A **random seed** is not a model-quality setting. It helps make experiments reproducible.

---

## `epochs` — How many times does the model see the training set?

One **epoch** = one complete pass through all training examples.

```text
Epoch 1: see every training sequence once
Epoch 2: see every training sequence again
Epoch 3: ...
```

**Increasing epochs can:**
- give the model more opportunities to learn,
- improve training fit,
- increase runtime,
- eventually increase the risk of overfitting.

**Too few epochs:** model may still be undertrained.

**Too many epochs:** training loss may continue falling while validation performance stops improving or gets worse.

This is why we graph metrics **across epochs** instead of only looking at the final value.

---

## `batch_size` — How many examples are processed before one update?

Suppose `batch_size = 8`.

The model:
1. processes 8 sequences,
2. combines their loss,
3. calculates gradients,
4. makes one optimizer update.

Then it moves to the next 8 sequences.

**Larger batches generally:**
- use more memory,
- make fewer optimizer updates per epoch,
- can improve hardware utilization,
- make gradient estimates less noisy.

**Smaller batches generally:**
- use less memory,
- make more updates per epoch,
- produce noisier gradient estimates.

A larger batch is **not automatically more accurate**.

On a GPU, the largest batch that fits in memory is not necessarily the best batch scientifically.

---

## `learning_rate` — How large is each optimizer step?

The learning rate controls the **size of weight updates**.

Think of optimization like walking downhill toward a lower-loss region:

```text
too small  → tiny steps → learning may be very slow
reasonable → useful steps → loss can improve steadily
too large  → huge steps → model may jump around or become unstable
```

Common values in this notebook might include:

```text
1e-5  = 0.00001
5e-5  = 0.00005
1e-4  = 0.00010
3e-4  = 0.00030
```

Because DNABERT is already pretrained, we often want **relatively small updates** so we adapt the model without immediately destroying useful pretrained information.

---

## `dropout` — Regularization during training

Dropout randomly turns off a fraction of activations during training.

For example:

```text
dropout = 0.0 → no dropout
dropout = 0.1 → about 10% dropped
dropout = 0.3 → about 30% dropped
```

The goal is to discourage the network from relying too heavily on particular internal features.

**Possible effect of increasing dropout:**
- may reduce overfitting,
- may improve generalization,
- but too much can make learning harder and cause underfitting.

Dropout behaves differently during validation: it is turned off when the model is in evaluation mode.

---

## `finetune_mode` — How much of pretrained DNABERT is allowed to change?

This is one of the most important experiments in this notebook.

### `"head_only"`

```text
DNABERT = frozen
Classifier = trainable
```

Only the new classification head learns.

**Advantages**
- fastest,
- fewest trainable parameters,
- preserves the pretrained DNABERT representation.

**Limitation**
- DNABERT itself cannot adapt its representations to CTCF binding.

### `"last_4"`

```text
Early DNABERT layers = frozen
Last 4 Transformer layers = trainable
Classifier = trainable
```

This is a middle ground.

The model can adapt some higher-level representations without updating the entire pretrained network.

### `"full"`

```text
All DNABERT layers = trainable
Classifier = trainable
```

This gives the model maximum flexibility.

**Advantages**
- greatest ability to specialize for CTCF.

**Trade-offs**
- more trainable parameters,
- more computation,
- potentially greater risk of overfitting or damaging useful pretrained representations if optimization is too aggressive.

---

## `max_length` — Maximum number of tokens sent to DNABERT

This is **token length**, not number of DNA bases.

Our 200-bp windows produce:

```text
200 - 6 + 1 = 195 overlapping 6-mers
```

DNABERT also adds special tokens, so `max_length = 256` is already enough for this dataset.

Increasing it to 512 here does **not** reveal more biological sequence because the original sequence is only 200 bp. It mainly creates more padding and can waste memory/computation.

If a future dataset contains longer DNA windows, then `max_length` becomes a meaningful experiment.

---

## `random_seed` — Reproducibility, not model power

Training includes random processes such as:
- initial classifier weights,
- shuffled training batches,
- dropout masks.

Using the same random seed helps students compare experiments more fairly.

```python
RANDOM_SEED = 42
```

does **not** mean 42 is a better seed.

For serious conclusions, researchers often repeat an experiment with several seeds and report the variation.

---

## A controlled experiment

If your baseline is:

```text
epochs = 2
batch = 8
learning rate = 1e-4
dropout = 0.1
mode = full
```

and you want to test learning rate, use:

```text
Run A: learning rate = 1e-5
Run B: learning rate = 5e-5
Run C: learning rate = 1e-4
Run D: learning rate = 3e-4
```

while keeping the other settings the same.

Then a graph of **learning rate vs AUROC** has a clear interpretation.

## Predict Before You Run

Before changing a parameter, write down:

1. **What are you changing?**
2. **What are you keeping fixed?**
3. **What do you predict will happen to AUROC?**
4. **What do you predict will happen to training time?**

Making the prediction first turns parameter tuning into an experiment rather than random trial-and-error.

In [ ]:
# ✏️ EDIT ME — YOUR FIRST EXPERIMENT
#
# Start by changing ONLY ONE value relative to the baseline.

EXPERIMENT_CONFIG = {"run_name": "experiment_1", "epochs": 2, "batch_size": 8,

    # Try: 1e-5, 5e-5, 1e-4, or 3e-4
    "learning_rate": 5e-5, "dropout": 0.1,

    # Choose: "head_only", "last_4", or "full"
    "finetune_mode": "full", "max_length": 256, "random_seed": RANDOM_SEED}

EXPERIMENT_CONFIG

In [ ]:
# ✏️ RUN THIS after making your prediction
experiment_model, experiment_history, experiment_result, experiment_predictions = (
    run_dnabert_experiment(**EXPERIMENT_CONFIG))

# 11. Explore Your Experimental Results

Your runs are saved as a normal table.

This means the same pandas tools you used for DNA exploration can now be used for **model-result exploration**.

In [ ]:
# 🔲 TRY IT — load all Notebook 1 experiments
results = load_notebook1_results()

results

In [ ]:
# 🔲 TRY IT — choose the metrics/parameters you want to compare
results[["run_name", "learning_rate", "epochs", "batch_size", "dropout",
        "finetune_mode", "final_val_accuracy", "final_val_precision",
        "final_val_recall", "final_val_f1", "best_val_auroc",
        "final_val_auprc", "total_training_time_seconds"]]

## Easy Graph Recipe 1 — Compare AUROC Across Runs

In [ ]:
# 👀 READ — bar graph from your own experiments
plt.figure(figsize=(8, 4))

plt.bar(results["run_name"], results["best_val_auroc"])

plt.axhline(0.5, linestyle="--", label="Random ranking")

plt.xlabel("Experiment")
plt.ylabel("Best validation AUROC")
plt.title("DNABERT Fine-Tuning Experiments")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.show()

## Easy Graph Recipe — Compare Any Validation Metric

Once you have several runs, you can reuse the same plotting pattern for different metrics.

Try changing:

```python
METRIC_TO_COMPARE = "final_val_accuracy"
```

to:

- `"final_val_precision"`
- `"final_val_recall"`
- `"final_val_f1"`
- `"best_val_auroc"`
- `"final_val_auprc"`

In [ ]:
# ✏️ EDIT ME
METRIC_TO_COMPARE = "best_val_auroc"

plt.figure(figsize=(8, 4))

plt.bar(results["run_name"], results[METRIC_TO_COMPARE])

plt.xlabel("Experiment")
plt.ylabel(METRIC_TO_COMPARE)
plt.title(f"{METRIC_TO_COMPARE} by Experiment")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Easy Graph Recipe 2 — Compare Training Time

In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(results["run_name"], results["total_training_time_seconds"])

plt.xlabel("Experiment")
plt.ylabel("Training time (seconds)")
plt.title("Training Time by Experiment")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Easy Graph Recipe 3 — Performance vs. Compute

A scatter plot can show whether spending more training time actually gave better validation performance.

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(results["total_training_time_seconds"], results["best_val_auroc"])

for _, row in results.iterrows():
    plt.annotate(row["run_name"], (row["total_training_time_seconds"],
            row["best_val_auroc"]))

plt.xlabel("Training time (seconds)")
plt.ylabel("Best validation AUROC")
plt.title("Performance vs. Training Time")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Easy Graph Recipe 4 — Explore One Parameter

Change `PARAMETER_TO_PLOT` to one of:

- `"learning_rate"`
- `"epochs"`
- `"batch_size"`
- `"dropout"`
- `"trainable_parameters"`

This graph is most meaningful when the other important settings were held constant.

In [ ]:
# ✏️ EDIT ME
PARAMETER_TO_PLOT = "learning_rate"

plot_df = results.sort_values(PARAMETER_TO_PLOT)

plt.figure(figsize=(7, 5))

plt.scatter(plot_df[PARAMETER_TO_PLOT], plot_df["best_val_auroc"])

for _, row in plot_df.iterrows():
    plt.annotate(row["run_name"], (row[PARAMETER_TO_PLOT],
            row["best_val_auroc"]))

plt.xlabel(PARAMETER_TO_PLOT)
plt.ylabel("Best validation AUROC")
plt.title(f"{PARAMETER_TO_PLOT} vs. Validation AUROC")

if PARAMETER_TO_PLOT == "learning_rate":
    plt.xscale("log")

plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# 12. Your Exploration Space

At this point you know enough to make your own controlled fine-tuning experiment.

A useful sequence is:

### Experiment A — Learning rate
Keep everything fixed and compare, for example:

```text
1e-5
5e-5
1e-4
3e-4
```

### Experiment B — How much of DNABERT should train?
Keep the same hyperparameters and compare:

```text
head_only
last_4
full
```

### Experiment C — Training length
Compare:

```text
1 epoch
2 epochs
4 epochs
```

You do **not** need to complete every possible combination in Notebook 1.

Notebook 3 will later give you HPC tools for running larger experiments.

## Questions You Can Ask the Data Yourself

You now have two datasets to explore:

### Biological dataset

```python
clean_df
```

Try questions such as:

```python
clean_df["gc_content"].mean()

clean_df.groupby("label_name")["gc_content"].mean()

clean_df["length"].value_counts()

clean_df.sample(5)
```

### Experiment dataset

```python
results
```

Try:

```python
results["best_val_auroc"].max()

results.sort_values("best_val_auroc", ascending=False)

results.groupby("finetune_mode")["best_val_auroc"].mean()

results[["run_name", "best_val_auroc", "total_training_time_seconds"]]
```

Exploration is not limited to the questions already written in the notebook.

# 12B. A Simple Recipe for Making Your Own Graph

Most of the plots in this notebook follow the same four-step pattern.

## 1. Choose the data

```python
x = results["run_name"]
y = results["best_val_auroc"]
```

## 2. Choose the graph type

For categories:

```python
plt.bar(x, y)
```

For a numeric relationship:

```python
plt.scatter(x, y)
```

For values changing across epochs:

```python
plt.plot(x, y)
```

## 3. Label it

```python
plt.xlabel("Experiment")
plt.ylabel("AUROC")
plt.title("My Experiment")
```

## 4. Show it

```python
plt.tight_layout()
plt.show()
```

### Which graph should I use?

| Question | Good first graph |
|---|---|
| Compare categories/runs | Bar graph |
| See a distribution | Histogram |
| Compare distributions between groups | Box plot |
| Follow training across epochs | Line graph |
| Compare two numeric variables | Scatter plot |
| Inspect classification mistakes | Confusion matrix |
| Evaluate thresholds | ROC or Precision–Recall curve |

You can copy one of the notebook's graph cells, change the columns, labels, and title, and build a new figure without starting from a blank screen.

# 13. Notebook 1 Summary

You have now followed the complete workflow:

```mermaid
flowchart LR
    A["DNA dataset"] --> B["Explore with pandas"]
    B --> C["Graph + audit"]
    C --> D["6-mer tokenizer"]
    D --> E["Pretrained DNABERT"]
    E --> F["Classification head"]
    F --> G["Fine-tune"]
    G --> H["Save results"]
    H --> I["Graph + compare experiments"]
```

### What you should carry into Notebook 2

You have **used** a Transformer and fine-tuned it.

In Notebook 2, you will start opening the black box and coding the essential Transformer pieces yourself:

**Q/K/V → attention → multi-head attention → feed-forward network → Transformer block → complete DNA classifier**

# 📚 Mini Glossary

| Term | Meaning |
|---|---|
| **DataFrame** | pandas table of rows and columns |
| **EDA** | exploratory data analysis |
| **GC content** | fraction of a DNA sequence made of G or C |
| **k-mer** | DNA substring of length `k` |
| **Token** | unit given to a model |
| **Token ID** | integer index representing a vocabulary token |
| **Embedding** | learned numerical vector representing a token |
| **Self-attention** | lets token representations use information from other tokens |
| **[CLS]** | special token whose final representation can summarize a sequence for classification |
| **Classification head** | new layer mapping model representation to class scores |
| **Fine-tuning** | adapting a pretrained model to a new task |
| **Epoch** | one pass through the training dataset |
| **Batch size** | examples processed before an optimizer update |
| **Learning rate** | size of optimization updates |
| **Dropout** | regularization that randomly drops activations during training |
| **Validation set** | held-out examples used to measure model performance |
| **AUROC** | ranking-quality metric for binary classification |